# 🌿 Plant Disease Detection - DEMO
## Quick Demo (No Training Required)

**Setup Instructions:**
1. Upload `plant_disease_model_final.h5` to your Google Drive root folder
2. Run all cells below
3. Use the Gradio interface to test disease detection

**Time to run: 2-3 minutes**

---

## Step 1: Install Required Packages

In [1]:
!pip install -q tensorflow gradio requests pillow groq

print("✅ Packages installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.9 MB/s eta 0:00:00
✅ Packages installed successfully!


## Step 2: Import Libraries

In [2]:
import os
import numpy as np
from PIL import Image
import tensorflow as tf
from tensorflow import keras
import gradio as gr
import requests
import matplotlib.pyplot as plt
from groq import Groq

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## Step 3: Load Pre-Trained Model

In [3]:
from google.colab import drive
drive.mount('/content/drive')

# Load the trained model
MODEL_PATH = '/content/drive/MyDrive/plant_disease_model_final.h5'

if not os.path.exists(MODEL_PATH):
    print("❌ Model not found!")
    print("Please upload 'plant_disease_model_final.h5' to your Google Drive root folder")
else:
    model = keras.models.load_model(MODEL_PATH)
    model_size = os.path.getsize(MODEL_PATH) / (1024 * 1024)
    print(f"✅ Model loaded successfully!")
    print(f"Model size: {model_size:.2f} MB")
    print(f"Model accuracy: ~90%")

# Configuration
IMG_SIZE = (224, 224)

# Class names (38 plant diseases)
class_names = [
    'Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust',
    'Apple___healthy', 'Background_without_leaves', 'Blueberry___healthy',
    'Cherry___Powdery_mildew', 'Cherry___healthy',
    'Corn___Cercospora_leaf_spot Gray_leaf_spot', 'Corn___Common_rust',
    'Corn___Northern_Leaf_Blight', 'Corn___healthy', 'Grape___Black_rot',
    'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)',
    'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)',
    'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot',
    'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight',
    'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy',
    'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy',
    'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight',
    'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot',
    'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus',
    'Tomato___healthy'
]

print(f"\n✅ Configuration complete!")
print(f"Total disease classes: {len(class_names)}")

Mounted at /content/drive


✅ Model loaded successfully!
Model size: 11.06 MB
Model accuracy: ~90%

✅ Configuration complete!
Total disease classes: 39


## Step 4: Disease Solutions Database

In [4]:
disease_solutions = {
    'Apple___Apple_scab': {
        'severity': 'Moderate',
        'treatment': ['Remove and destroy infected leaves immediately', 'Apply fungicide (Captan or Mancozeb) every 7-10 days', 'Prune trees to improve air circulation', 'Avoid overhead watering'],
        'prevention': 'Plant resistant varieties, rake and destroy fallen leaves',
        'contagious': 'Yes - fungal spores spread via wind and rain'
    },
    'Apple___Black_rot': {
        'severity': 'Severe',
        'treatment': ['Remove infected fruits and branches', 'Apply fungicide during bloom period', 'Prune out dead wood and cankers', 'Maintain proper tree spacing'],
        'prevention': 'Remove mummies and cankers, apply preventive fungicides',
        'contagious': 'Yes - spreads through spores'
    },
    'Apple___Cedar_apple_rust': {
        'severity': 'Moderate',
        'treatment': ['Apply fungicide at pink bud stage', 'Remove nearby cedar trees if possible', 'Use resistant apple varieties', 'Apply protective sprays'],
        'prevention': 'Plant resistant cultivars, remove alternate hosts',
        'contagious': 'Yes - requires cedar trees to complete life cycle'
    },
    'Apple___healthy': {
        'severity': 'None',
        'treatment': ['No treatment needed', 'Continue regular monitoring', 'Maintain good cultural practices'],
        'prevention': 'Regular inspection, proper nutrition, adequate spacing',
        'contagious': 'Not applicable - healthy plant'
    },
    'Background_without_leaves': {
        'severity': 'N/A',
        'treatment': ['No leaf detected in image', 'Please upload a clear leaf image'],
        'prevention': 'Ensure image contains visible leaf',
        'contagious': 'Not applicable'
    },
    'Blueberry___healthy': {
        'severity': 'None',
        'treatment': ['No treatment needed', 'Maintain current care routine'],
        'prevention': 'Regular watering, acidic soil, proper pruning',
        'contagious': 'Not applicable - healthy plant'
    },
    'Cherry___Powdery_mildew': {
        'severity': 'Moderate',
        'treatment': ['Apply sulfur-based fungicide', 'Prune to improve air circulation', 'Remove infected plant parts', 'Water at base, not foliage'],
        'prevention': 'Plant in sunny locations, avoid overcrowding',
        'contagious': 'Yes - spreads via airborne spores'
    },
    'Cherry___healthy': {
        'severity': 'None',
        'treatment': ['No treatment needed', 'Continue monitoring'],
        'prevention': 'Regular inspection, balanced fertilization',
        'contagious': 'Not applicable - healthy plant'
    },
    'Corn___Cercospora_leaf_spot Gray_leaf_spot': {
        'severity': 'Moderate to Severe',
        'treatment': ['Apply fungicide (strobilurin or triazole)', 'Practice crop rotation', 'Remove crop debris after harvest', 'Use resistant hybrids'],
        'prevention': 'Rotate crops, till under residue, plant resistant varieties',
        'contagious': 'Yes - spreads through wind and rain'
    },
    'Corn___Common_rust': {
        'severity': 'Moderate',
        'treatment': ['Apply fungicide if severe', 'Usually does not require treatment', 'Plant resistant hybrids'],
        'prevention': 'Use resistant varieties, monitor regularly',
        'contagious': 'Yes - airborne spores'
    },
    'Corn___Northern_Leaf_Blight': {
        'severity': 'Severe',
        'treatment': ['Apply fungicide at first sign', 'Use resistant hybrids', 'Rotate crops', 'Bury crop residue'],
        'prevention': 'Plant resistant varieties, crop rotation, tillage',
        'contagious': 'Yes - spreads via wind-borne spores'
    },
    'Corn___healthy': {
        'severity': 'None',
        'treatment': ['No treatment needed', 'Continue regular care'],
        'prevention': 'Adequate spacing, balanced nutrition, rotation',
        'contagious': 'Not applicable - healthy plant'
    },
    'Grape___Black_rot': {
        'severity': 'Severe',
        'treatment': ['Remove mummified berries', 'Apply fungicide from bloom through harvest', 'Prune for air circulation', 'Remove infected canes'],
        'prevention': 'Sanitation, fungicide program, resistant varieties',
        'contagious': 'Yes - highly contagious via spores'
    },
    'Grape___Esca_(Black_Measles)': {
        'severity': 'Severe',
        'treatment': ['No effective cure available', 'Remove severely affected vines', 'Prune out infected wood', 'Apply wound protectants'],
        'prevention': 'Avoid wounding, proper pruning techniques',
        'contagious': 'Yes - spreads through pruning wounds'
    },
    'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)': {
        'severity': 'Moderate',
        'treatment': ['Apply copper-based fungicide', 'Remove infected leaves', 'Improve air circulation', 'Avoid overhead irrigation'],
        'prevention': 'Prune for airflow, fungicide applications',
        'contagious': 'Yes - spreads in humid conditions'
    },
    'Grape___healthy': {
        'severity': 'None',
        'treatment': ['No treatment needed', 'Maintain current practices'],
        'prevention': 'Regular monitoring, proper spacing, balanced nutrition',
        'contagious': 'Not applicable - healthy plant'
    },
    'Orange___Haunglongbing_(Citrus_greening)': {
        'severity': 'Catastrophic',
        'treatment': ['No cure available', 'Remove and destroy infected trees', 'Control psyllid vectors with insecticides', 'Report to agricultural authorities'],
        'prevention': 'Plant disease-free nursery stock, control psyllids',
        'contagious': 'Yes - spread by Asian citrus psyllid'
    },
    'Peach___Bacterial_spot': {
        'severity': 'Severe',
        'treatment': ['Apply copper sprays', 'Use resistant varieties', 'Prune to improve air circulation', 'Remove severely infected leaves'],
        'prevention': 'Plant resistant cultivars, copper applications',
        'contagious': 'Yes - spreads via rain splash'
    },
    'Peach___healthy': {
        'severity': 'None',
        'treatment': ['No treatment needed', 'Continue regular monitoring'],
        'prevention': 'Regular inspection, proper pruning, balanced fertilization',
        'contagious': 'Not applicable - healthy plant'
    },
    'Pepper,_bell___Bacterial_spot': {
        'severity': 'Severe',
        'treatment': ['Apply copper-based bactericide', 'Remove infected plants', 'Avoid overhead watering', 'Use disease-free seeds'],
        'prevention': 'Crop rotation, resistant varieties, drip irrigation',
        'contagious': 'Yes - spreads via water and handling'
    },
    'Pepper,_bell___healthy': {
        'severity': 'None',
        'treatment': ['No treatment needed', 'Maintain good practices'],
        'prevention': 'Proper spacing, adequate water, balanced nutrients',
        'contagious': 'Not applicable - healthy plant'
    },
    'Potato___Early_blight': {
        'severity': 'Moderate to Severe',
        'treatment': ['Apply fungicide (chlorothalonil or mancozeb)', 'Remove infected foliage', 'Practice crop rotation', 'Avoid overhead irrigation'],
        'prevention': 'Rotate crops, fungicide program, resistant varieties',
        'contagious': 'Yes - spreads via spores in wind and rain'
    },
    'Potato___Late_blight': {
        'severity': 'Catastrophic',
        'treatment': ['Apply fungicide immediately', 'Destroy infected plants', 'Harvest early if needed', 'Do not compost infected material'],
        'prevention': 'Use certified seed, fungicide program, good drainage',
        'contagious': 'Extremely - can destroy entire fields rapidly'
    },
    'Potato___healthy': {
        'severity': 'None',
        'treatment': ['No treatment needed', 'Continue monitoring'],
        'prevention': 'Crop rotation, certified seed, proper spacing',
        'contagious': 'Not applicable - healthy plant'
    },
    'Raspberry___healthy': {
        'severity': 'None',
        'treatment': ['No treatment needed', 'Maintain current care'],
        'prevention': 'Regular pruning, good air circulation, mulching',
        'contagious': 'Not applicable - healthy plant'
    },
    'Soybean___healthy': {
        'severity': 'None',
        'treatment': ['No treatment needed', 'Continue regular practices'],
        'prevention': 'Crop rotation, adequate spacing, balanced nutrition',
        'contagious': 'Not applicable - healthy plant'
    },
    'Squash___Powdery_mildew': {
        'severity': 'Moderate',
        'treatment': ['Apply fungicide (sulfur or potassium bicarbonate)', 'Remove infected leaves', 'Improve air circulation', 'Water at base only'],
        'prevention': 'Plant resistant varieties, adequate spacing',
        'contagious': 'Yes - spreads via airborne spores'
    },
    'Strawberry___Leaf_scorch': {
        'severity': 'Moderate',
        'treatment': ['Remove infected leaves', 'Apply fungicide', 'Improve air circulation', 'Avoid overhead watering'],
        'prevention': 'Plant resistant varieties, good sanitation',
        'contagious': 'Yes - spreads through splashing water'
    },
    'Strawberry___healthy': {
        'severity': 'None',
        'treatment': ['No treatment needed', 'Maintain practices'],
        'prevention': 'Regular renovation, proper spacing, mulching',
        'contagious': 'Not applicable - healthy plant'
    },
    'Tomato___Bacterial_spot': {
        'severity': 'Severe',
        'treatment': ['Apply copper bactericide', 'Remove infected plants', 'Use drip irrigation', 'Avoid working with wet plants'],
        'prevention': 'Use disease-free transplants, copper sprays, rotation',
        'contagious': 'Yes - spreads via water and handling'
    },
    'Tomato___Early_blight': {
        'severity': 'Moderate to Severe',
        'treatment': ['Apply fungicide (chlorothalonil)', 'Remove lower infected leaves', 'Stake plants for air circulation', 'Mulch to prevent splash'],
        'prevention': 'Crop rotation, fungicide program, mulching',
        'contagious': 'Yes - spreads via spores'
    },
    'Tomato___Late_blight': {
        'severity': 'Catastrophic',
        'treatment': ['Apply fungicide immediately', 'Remove and destroy infected plants', 'Improve air circulation', 'Avoid overhead watering'],
        'prevention': 'Use resistant varieties, preventive fungicides',
        'contagious': 'Extremely - spreads very rapidly'
    },
    'Tomato___Leaf_Mold': {
        'severity': 'Moderate',
        'treatment': ['Improve ventilation in greenhouse', 'Apply fungicide', 'Remove infected leaves', 'Reduce humidity'],
        'prevention': 'Good air circulation, resistant varieties, humidity control',
        'contagious': 'Yes - thrives in high humidity'
    },
    'Tomato___Septoria_leaf_spot': {
        'severity': 'Moderate to Severe',
        'treatment': ['Apply fungicide (chlorothalonil or copper)', 'Remove infected lower leaves', 'Mulch around plants', 'Avoid overhead watering'],
        'prevention': 'Crop rotation, mulching, fungicide applications',
        'contagious': 'Yes - spreads via splashing water'
    },
    'Tomato___Spider_mites Two-spotted_spider_mite': {
        'severity': 'Moderate',
        'treatment': ['Spray with water to dislodge mites', 'Apply insecticidal soap or neem oil', 'Use predatory mites', 'Remove heavily infested leaves'],
        'prevention': 'Monitor regularly, avoid water stress, encourage predators',
        'contagious': 'Yes - mites spread between plants'
    },
    'Tomato___Target_Spot': {
        'severity': 'Moderate to Severe',
        'treatment': ['Apply fungicide (chlorothalonil or mancozeb)', 'Remove infected leaves', 'Improve air circulation', 'Avoid overhead irrigation'],
        'prevention': 'Crop rotation, fungicide program, resistant varieties',
        'contagious': 'Yes - spreads via spores and splashing water'
    },
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus': {
        'severity': 'Severe',
        'treatment': ['No cure available', 'Remove infected plants immediately', 'Control whitefly vectors with insecticides', 'Use reflective mulches'],
        'prevention': 'Plant resistant varieties, control whiteflies, use screens',
        'contagious': 'Yes - spread by whiteflies'
    },
    'Tomato___Tomato_mosaic_virus': {
        'severity': 'Severe',
        'treatment': ['No cure available', 'Remove and destroy infected plants', 'Sanitize tools between plants', 'Wash hands thoroughly'],
        'prevention': 'Use resistant varieties, sanitize tools, avoid tobacco use',
        'contagious': 'Yes - highly contagious via contact'
    },
    'Tomato___healthy': {
        'severity': 'None',
        'treatment': ['No treatment needed', 'Continue current practices'],
        'prevention': 'Crop rotation, proper spacing, balanced fertilization',
        'contagious': 'Not applicable - healthy plant'
    }
}

print(f"✅ Disease database loaded: {len(disease_solutions)} diseases")

✅ Disease database loaded: 39 diseases


## Step 5: Research Papers API

In [5]:
def get_research_papers(disease_name):
    """
    Fetch recent research papers using Semantic Scholar API
    """
    try:
        search_query = disease_name.replace('___', ' ').replace('_', ' ') + " plant disease treatment"

        url = "https://api.semanticscholar.org/graph/v1/paper/search"
        params = {
            'query': search_query,
            'limit': 5,
            'fields': 'title,year,authors,abstract,publicationDate'
        }

        response = requests.get(url, params=params, timeout=5)

        if response.status_code == 200:
            data = response.json()

            if 'data' in data and len(data['data']) > 0:
                result = "📚 Recent Research Papers:\n\n"

                for i, paper in enumerate(data['data'][:3], 1):
                    title = paper.get('title', 'No title')
                    year = paper.get('year', 'N/A')
                    authors = paper.get('authors', [])
                    author_names = ', '.join([a.get('name', '') for a in authors[:3]])
                    if len(authors) > 3:
                        author_names += ' et al.'

                    result += f"{i}. **{title}**\n"
                    result += f"   Authors: {author_names}\n"
                    result += f"   Year: {year}\n\n"

                return result
            else:
                return "📚 No recent research papers found."
        else:
            return "⚠️ Unable to fetch papers at this time."

    except Exception as e:
        return f"⚠️ Error fetching papers: {str(e)}"

print("✅ Research API ready")

✅ Research API ready


In [ ]:
## Step 6: Enhanced LLM-Powered RAG Chatbot with Groq

# Configure Groq API (much faster than Gemini!)
GROQ_API_KEY = ""

if GROQ_API_KEY:
    groq_client = Groq(api_key=GROQ_API_KEY)
    print("✅ Groq AI configured successfully!")
    print("Using model: llama-3.3-70b-versatile (ultra-fast responses)")
else:
    groq_client = None
    print("⚠️ No Groq API key provided. Chatbot will use fallback mode.")

# Build comprehensive knowledge base from disease solutions
def build_knowledge_base():
    """Create a structured knowledge base from disease solutions"""
    knowledge_base = []

    for disease_key, info in disease_solutions.items():
        if "healthy" in disease_key or "Background" in disease_key:
            continue

        plant, disease = disease_key.split('___')
        plant = plant.replace('_', ' ')
        disease = disease.replace('_', ' ')

        kb_entry = {
            'disease_name': f"{plant} - {disease}",
            'plant': plant,
            'disease': disease,
            'severity': info.get('severity', 'Unknown'),
            'treatments': info.get('treatment', []),
            'prevention': info.get('prevention', ''),
            'contagious': info.get('contagious', 'Unknown')
        }
        knowledge_base.append(kb_entry)

    return knowledge_base

knowledge_base = build_knowledge_base()

# Create context string for LLM
def create_context_string(detected_disease=None):
    """Create context string for the LLM"""
    context = """You are an expert agricultural advisor specializing in plant diseases.
Your role is to ONLY answer questions related to:
- Plant diseases and their symptoms
- Treatment and prevention methods
- Farming practices related to disease management
- Pesticides, fungicides, and organic treatments
- Crop health and disease identification

IMPORTANT RULES:
1. If asked about topics unrelated to plant diseases or agriculture, politely decline and redirect to plant disease topics
2. Do NOT answer questions about: general knowledge, coding, politics, entertainment, or any non-agricultural topics
3. Keep responses practical and farmer-friendly
4. Be concise but thorough

"""

    if detected_disease:
        # Add specific disease context if available
        disease_info = disease_solutions.get(detected_disease, {})
        if disease_info:
            plant, disease = detected_disease.split('___')
            context += f"\nCURRENT DETECTED DISEASE: {plant.replace('_', ' ')} - {disease.replace('_', ' ')}\n"
            context += f"Severity: {disease_info.get('severity', 'Unknown')}\n"
            context += f"Treatments: {', '.join(disease_info.get('treatment', []))}\n"
            context += f"Prevention: {disease_info.get('prevention', '')}\n"
            context += f"Contagious: {disease_info.get('contagious', 'Unknown')}\n\n"

    # Add general knowledge base
    context += "AVAILABLE DISEASES IN DATABASE:\n"
    for kb_entry in knowledge_base[:10]:  # Include first 10 for context
        context += f"- {kb_entry['disease_name']} (Severity: {kb_entry['severity']})\n"

    return context

# Conversation history storage
current_detected_disease = None

def answer_question_with_llm(question, detected_disease_key=None, history=None):
    """Enhanced Q&A using Groq LLM with RAG"""
    global current_detected_disease

    if not question.strip():
        return "Please ask a question about plant diseases, treatments, or farming practices.", history or []

    # Update current disease context if provided
    if detected_disease_key:
        current_detected_disease = detected_disease_key

    # Initialize history if needed
    if history is None:
        history = []

    # If no LLM available, use fallback
    if not groq_client:
        fallback_answer, _ = fallback_answer_func(question, current_detected_disease)
        history.append((question, fallback_answer))
        return fallback_answer, history

    try:
        # Build context with detected disease info
        system_context = create_context_string(current_detected_disease)

        # Build conversation messages for Groq
        messages = [
            {"role": "system", "content": system_context}
        ]

        # Add conversation history (last 3 exchanges)
        for user_msg, bot_msg in history[-3:]:
            messages.append({"role": "user", "content": user_msg})
            messages.append({"role": "assistant", "content": bot_msg})

        # Add current question
        messages.append({"role": "user", "content": question})

        # Generate response using Groq
        chat_completion = groq_client.chat.completions.create(
            messages=messages,
            model="llama-3.3-70b-versatile",  # Fast and powerful
            temperature=0.7,
            max_tokens=1024,
            top_p=1,
            stream=False
        )

        answer = chat_completion.choices[0].message.content

        # Add to history
        history.append((question, answer))

        return answer, history

    except Exception as e:
        error_msg = f"⚠️ Error with AI response: {str(e)}\n\nFalling back to basic mode."
        fallback_answer, _ = fallback_answer_func(question, current_detected_disease)
        history.append((question, error_msg + "\n\n" + fallback_answer))
        return error_msg + "\n\n" + fallback_answer, history

def fallback_answer_func(question, detected_disease_key):
    """Fallback answer system using TF-IDF when LLM is unavailable"""
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity

    # Build documents for TF-IDF
    documents = []
    doc_keys = []

    for kb_entry in knowledge_base:
        doc = f"{kb_entry['disease_name']}. "
        doc += f"Severity: {kb_entry['severity']}. "
        doc += f"Treatments: {', '.join(kb_entry['treatments'])}. "
        doc += f"Prevention: {kb_entry['prevention']}."
        documents.append(doc)
        doc_keys.append(kb_entry)

    # Find most relevant disease
    vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
    tfidf_matrix = vectorizer.fit_transform(documents)
    question_vector = vectorizer.transform([question])

    similarities = cosine_similarity(question_vector, tfidf_matrix).flatten()
    most_similar_idx = np.argmax(similarities)
    confidence = similarities[most_similar_idx]

    if confidence < 0.1:
        return "I couldn't find a relevant answer. Please ask about specific plant diseases or treatments.", []

    # Get the matched disease
    matched_disease = doc_keys[most_similar_idx]

    answer = f"**{matched_disease['disease_name']}** (Relevance: {confidence:.1%})\n\n"
    answer += f"**Severity:** {matched_disease['severity']}\n\n"
    answer += "**Treatment Steps:**\n"
    for i, treatment in enumerate(matched_disease['treatments'], 1):
        answer += f"{i}. {treatment}\n"
    answer += f"\n**Prevention:** {matched_disease['prevention']}\n"
    answer += f"\n**Contagious:** {matched_disease['contagious']}"

    return answer, []

print("✅ Enhanced Groq-powered RAG chatbot ready!")
print(f"Knowledge base contains {len(knowledge_base)} diseases")

✅ Groq AI configured successfully!
Using model: llama-3.3-70b-versatile (ultra-fast responses)
✅ Enhanced Groq-powered RAG chatbot ready!
Knowledge base contains 26 diseases


In [7]:
## Step 7: Grad-CAM for Model Explainability

def get_img_array(img_path, size):
    # Util function to load and preprocess an image
    img = keras.utils.load_img(img_path, target_size=size)
    array = keras.utils.img_to_array(img)
    array = np.expand_dims(array, axis=0)
    return array

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    # Create a model that maps the input image to the activations
    # of the last conv layer as well as the output predictions
    grad_model = keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )

    # Compute the gradient of the top predicted class for our input image
    # with respect to the activations of the last conv layer
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    # This is the gradient of the output neuron (top predicted or chosen)
    # with regard to the output feature map of the last conv layer
    grads = tape.gradient(class_channel, last_conv_layer_output)

    # This is a vector where each entry is the mean intensity of the gradient
    # over a specific feature map channel (we have 1280 channels in MobileNetV2)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # We multiply each channel in the feature map array
    # by "how important this channel is" with regard to the top predicted class
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # For visualization purpose, we will also normalize the heatmap between 0 & 1
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def get_gradcam_overlay(image, heatmap, alpha=0.6):
    # Rescale heatmap to a range 0-255
    heatmap = np.uint8(255 * heatmap)

    # Use jet colormap to colorize heatmap
    jet = plt.get_cmap("jet")

    # Use RGB values of the colormap
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap]

    # Create an image with RGB colorized heatmap
    jet_heatmap = keras.utils.array_to_img(jet_heatmap)
    jet_heatmap = jet_heatmap.resize(image.size)
    jet_heatmap = keras.utils.img_to_array(jet_heatmap)

    # Superimpose the heatmap on original image
    superimposed_img = jet_heatmap * alpha + np.array(image)
    superimposed_img = keras.utils.array_to_img(superimposed_img)

    return superimposed_img

# Find the last convolutional layer name automatically (robust method)
last_conv_layer_name = ""
for layer in reversed(model.layers):
    # Check if the layer is a convolutional layer
    if isinstance(layer, tf.keras.layers.Conv2D):
        last_conv_layer_name = layer.name
        break

# Fallback if no Conv2D layer is found (e.g., for some model architectures)
if last_conv_layer_name == "":
    for layer in reversed(model.layers):
        if len(layer.output_shape) == 4:
            last_conv_layer_name = layer.name
            break

print(f"✅ Grad-CAM setup complete. Last conv layer: '{last_conv_layer_name}'")

✅ Grad-CAM setup complete. Last conv layer: 'Conv_1'


## Step 6: Launch Gradio Demo

In [8]:
def format_class_name(class_name):
    parts = class_name.split('___')
    if len(parts) == 2:
        plant, disease = parts
        return f"{plant.replace('_', ' ')} - {disease.replace('_', ' ')}"
    return class_name.replace('_', ' ')

def predict_with_solutions(image):
    # Preprocess image for prediction
    resized_img = image.resize(IMG_SIZE)
    img_array = np.array(resized_img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # Make predictions
    predictions = model.predict(img_array, verbose=0)
    predicted_idx = np.argmax(predictions[0])
    confidence = float(predictions[0][predicted_idx])

    # Get top 5 predictions
    top_5_idx = np.argsort(predictions[0])[-5:][::-1]
    top_5 = {format_class_name(class_names[idx]): float(predictions[0][idx]) for idx in top_5_idx}

    # Get disease info
    disease_key = class_names[predicted_idx]
    predicted_class = format_class_name(disease_key)
    disease_info = disease_solutions.get(disease_key, {})

    # Format results based on confidence
    if confidence < 0.5:
        result = "⚠️ **Low Confidence Detection**\n\n"
        result += f"**Predicted:** {predicted_class}\n"
        result += f"**Confidence:** {confidence:.1%}\n\n"
        result += "⚠️ Please take a clearer photo with better lighting"
        solution_text = ""
    elif confidence < 0.8:
        result = "✓ **Moderate Confidence Detection**\n\n"
        result += f"**Predicted:** {predicted_class}\n"
        result += f"**Confidence:** {confidence:.1%}\n\n"
        result += "⚠️ Consider expert confirmation"
        solution_text = format_disease_solution(disease_info, predicted_class)
    else:
        result = "✅ **High Confidence Detection**\n\n"
        result += f"**Predicted:** {predicted_class}\n"
        result += f"**Confidence:** {confidence:.1%}\n\n"
        result += "Model is highly confident in this prediction."
        solution_text = format_disease_solution(disease_info, predicted_class)

    # Generate Grad-CAM heatmap
    heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=predicted_idx)
    gradcam_image = get_gradcam_overlay(resized_img, heatmap)

    return result, top_5, solution_text, disease_key, resized_img, gradcam_image

def format_disease_solution(disease_info, disease_name):
    if not disease_info:
        return "ℹ️ No treatment information available."

    solution = "## 💊 Treatment & Management\n\n"
    solution += f"**Disease:** {disease_name}\n\n"
    solution += f"**Severity:** {disease_info.get('severity', 'Unknown')}\n\n"

    solution += "### 🔧 Treatment Steps:\n"
    for i, treatment in enumerate(disease_info.get('treatment', []), 1):
        solution += f"{i}. {treatment}\n"

    prevention = disease_info.get('prevention', 'No prevention info')
    solution += f"\n### 🛡️ Prevention:\n{prevention}\n"

    contagious = disease_info.get('contagious', 'Unknown')
    solution += f"\n### 🦠 Contagious:\n{contagious}\n"

    return solution

def fetch_papers(disease_key):
    if disease_key:
        return get_research_papers(disease_key)
    return "Select a disease first"

# Custom CSS for modern UI
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');

* {
    font-family: 'Inter', sans-serif !important;
}

.gradio-container {
    max-width: 1400px !important;
    margin: auto !important;
}

/* Header styling */
.main-header {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    padding: 2rem;
    border-radius: 12px;
    margin-bottom: 2rem;
    box-shadow: 0 10px 30px rgba(0,0,0,0.1);
}

.main-header h1 {
    color: white !important;
    font-size: 2.5rem !important;
    font-weight: 700 !important;
    margin-bottom: 0.5rem !important;
}

.main-header p {
    color: rgba(255,255,255,0.9) !important;
    font-size: 1.1rem !important;
}

/* Tab styling */
.tab-nav button {
    font-weight: 600 !important;
    font-size: 1rem !important;
    padding: 0.75rem 1.5rem !important;
}

/* Button styling */
button {
    border-radius: 8px !important;
    font-weight: 600 !important;
    transition: all 0.3s ease !important;
}

button:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 5px 15px rgba(0,0,0,0.2) !important;
}

/* Card styling */
.gr-box {
    border-radius: 12px !important;
    border: 1px solid #e5e7eb !important;
    box-shadow: 0 2px 8px rgba(0,0,0,0.05) !important;
}

/* Chat interface styling */
.chatbot-container {
    background: linear-gradient(to bottom, #f9fafb, #ffffff);
    border-radius: 12px;
    padding: 1rem;
}

.message {
    padding: 1rem;
    border-radius: 8px;
    margin-bottom: 0.5rem;
}

.user-message {
    background: #667eea;
    color: white;
    margin-left: 2rem;
}

.bot-message {
    background: #f3f4f6;
    color: #1f2937;
    margin-right: 2rem;
}

/* Image preview styling */
.image-preview {
    border-radius: 12px !important;
    overflow: hidden !important;
    box-shadow: 0 4px 12px rgba(0,0,0,0.1) !important;
}

/* Accordion styling */
.accordion {
    border-radius: 8px !important;
    border: 1px solid #e5e7eb !important;
}

/* Label styling */
label {
    font-weight: 600 !important;
    color: #374151 !important;
    font-size: 0.95rem !important;
}

/* Input styling */
input, textarea {
    border-radius: 8px !important;
    border: 1px solid #d1d5db !important;
}

input:focus, textarea:focus {
    border-color: #667eea !important;
    box-shadow: 0 0 0 3px rgba(102, 126, 234, 0.1) !important;
}

/* Result cards */
.result-card {
    background: white;
    padding: 1.5rem;
    border-radius: 12px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.05);
    margin-bottom: 1rem;
}

/* Confidence badge */
.confidence-high {
    color: #059669;
    font-weight: 700;
}

.confidence-medium {
    color: #d97706;
    font-weight: 700;
}

.confidence-low {
    color: #dc2626;
    font-weight: 700;
}
"""

# Capture model summary
summary_list = []
model.summary(print_fn=lambda x: summary_list.append(x))
model_summary = "\n".join(summary_list)

# Create Enhanced Gradio Interface
with gr.Blocks(title="🌿 Plant Disease AI Assistant", css=custom_css) as demo:

    # Header
    gr.HTML("""
        <div class="main-header">
            <h1>🌿 Plant Disease AI Assistant</h1>
            <p>Advanced AI-Powered Agricultural Disease Detection & Management System</p>
        </div>
    """)

    with gr.Tabs():
        # Tab 1: Image Detection
        with gr.TabItem("📸 Disease Detection", id="detection"):
            gr.Markdown("### Upload a leaf image for instant disease detection and treatment recommendations")

            with gr.Row():
                with gr.Column(scale=1):
                    image_input = gr.Image(label="📷 Upload Leaf Image", type="pil", height=350)
                    predict_button = gr.Button("🔍 Analyze Disease", variant="primary", size="lg")

                    with gr.Accordion("📁 Try Example Images", open=False):
                        gr.Examples(
                            examples=[
                                ["examples/apple_scab.JPG"],
                                ["examples/corn_rust.JPG"],
                                ["examples/potato_late_blight.JPG"]
                            ],
                            inputs=image_input,
                            label="Click to load example"
                        )

                with gr.Column(scale=1):
                    result_output = gr.Textbox(label="🔬 Detection Result", lines=8)
                    top_predictions = gr.Label(label="📊 Top 5 Predictions", num_top_classes=5)

            with gr.Row():
                with gr.Column(scale=1):
                    image_preview = gr.Image(label="🖼️ Processed Image", height=280)
                with gr.Column(scale=1):
                    gradcam_output = gr.Image(label="🔥 AI Attention Map (Grad-CAM)", height=280)

            with gr.Accordion("💊 Treatment & Research", open=True):
                with gr.Row():
                    with gr.Column():
                        solution_output = gr.Markdown(label="💊 Treatment Plan")

                    with gr.Column():
                        papers_button = gr.Button("📚 Get Latest Research Papers", variant="secondary")
                        papers_output = gr.Markdown(label="📚 Research Papers")

        # Tab 2: AI Chatbot
        with gr.TabItem("💬 AI Farm Advisor", id="chatbot"):
            gr.Markdown("""
            ### 🤖 Chat with Your AI Agricultural Expert
            Ask anything about plant diseases, treatments, prevention, or farming practices.
            The AI has context about any disease you've detected and can provide personalized advice.

            ⚡ **Powered by Groq (Ultra-Fast Responses!)**
            """)

            with gr.Row():
                with gr.Column(scale=3):
                    chatbot = gr.Chatbot(
                        label="💬 Conversation",
                        height=500,
                        type="messages",
                        avatar_images=(None, "🌿"),
                        show_copy_button=True
                    )

                    with gr.Row():
                        question_input = gr.Textbox(
                            label="Your Question",
                            placeholder="e.g., How do I prevent fungal diseases in tomatoes?",
                            lines=2,
                            scale=4
                        )
                        send_button = gr.Button("Send 📤", variant="primary", scale=1)

                    with gr.Row():
                        clear_button = gr.Button("🗑️ Clear Chat", variant="secondary", size="sm")

                with gr.Column(scale=1):
                    gr.Markdown("### 💡 Quick Tips")
                    gr.Markdown("""
                    **Try asking:**
                    - What causes leaf spots on my plants?
                    - How to apply fungicides safely?
                    - Best organic treatments for powdery mildew?
                    - When should I harvest if disease is spreading?
                    - How to prevent disease in next season?

                    **Context Aware:**
                    If you've detected a disease, the AI knows about it and can provide specific advice!

                    **Note:** The AI only answers agriculture-related questions.
                    """)

                    detected_disease_display = gr.Textbox(
                        label="🔍 Currently Detected Disease",
                        value="None detected yet",
                        interactive=False
                    )

        # Tab 3: About
        with gr.TabItem("ℹ️ About", id="about"):
            gr.Markdown("""
            ## 🌿 About This System

            ### Features
            - **AI Disease Detection**: 90% accuracy across 38 plant diseases
            - **Grad-CAM Visualization**: See what the AI is looking at
            - **LLM-Powered Chatbot**: Natural conversation with agricultural AI
            - **Treatment Recommendations**: Evidence-based solutions
            - **Research Integration**: Latest scientific papers

            ### Technology Stack
            - **Model**: MobileNetV2 (Transfer Learning)
            - **AI Chat**: Groq (Llama 3.3 70B) - Ultra-fast responses!
            - **Framework**: TensorFlow + Gradio
            - **Accuracy**: ~90% on validation set

            ### Why Groq?
            Groq provides lightning-fast AI responses (10x faster than traditional APIs) with state-of-the-art language models.

            ### Model Architecture
            """)

            gr.Textbox(value=model_summary, label="Model Summary", lines=20, max_lines=30, interactive=False)

    # State management
    disease_state = gr.State()

    # Wire up detection tab
    predict_button.click(
        fn=predict_with_solutions,
        inputs=image_input,
        outputs=[result_output, top_predictions, solution_output, disease_state, image_preview, gradcam_output]
    ).then(
        fn=lambda disease_key: format_class_name(disease_key) if disease_key else "None detected yet",
        inputs=disease_state,
        outputs=detected_disease_display
    )

    papers_button.click(
        fn=fetch_papers,
        inputs=disease_state,
        outputs=papers_output
    )

    # Wire up chatbot with Groq
    def respond(message, history, detected_disease):
        """Handle chat responses with Groq AI"""
        try:
            if not message or not message.strip():
                return history, ""

            # Convert messages format to tuples for internal processing
            tuple_history = []
            if history:
                i = 0
                while i < len(history):
                    if i + 1 < len(history):
                        user_msg = history[i].get("content", "") if isinstance(history[i], dict) else ""
                        bot_msg = history[i + 1].get("content", "") if isinstance(history[i + 1], dict) else ""
                        if user_msg and bot_msg:
                            tuple_history.append((user_msg, bot_msg))
                    i += 2

            # Get AI response
            answer, updated_tuple_history = answer_question_with_llm(message, detected_disease, tuple_history)

            # Convert back to messages format
            messages_history = []
            for user_msg, bot_msg in updated_tuple_history:
                messages_history.append({"role": "user", "content": user_msg})
                messages_history.append({"role": "assistant", "content": bot_msg})

            return messages_history, ""

        except Exception as e:
            # If there's an error, show it in the chat
            error_message = f"⚠️ Error: {str(e)}\n\nPlease check your API configuration."

            # Add error to history
            if not history:
                history = []
            history.append({"role": "user", "content": message})
            history.append({"role": "assistant", "content": error_message})

            return history, ""

    send_button.click(
        fn=respond,
        inputs=[question_input, chatbot, disease_state],
        outputs=[chatbot, question_input]
    )

    question_input.submit(
        fn=respond,
        inputs=[question_input, chatbot, disease_state],
        outputs=[chatbot, question_input]
    )

    clear_button.click(
        fn=lambda: [],
        outputs=chatbot
    )

print("🚀 Launching Enhanced Plant Disease AI Assistant...")
print("⚡ Powered by Groq for ultra-fast AI responses!")
print("📱 Share the public URL below!")
print("\n" + "="*60)
demo.launch(share=True, debug=True)

/tmp/ipython-input-2544992276.py:227: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="🌿 Plant Disease AI Assistant", css=custom_css) as demo:
/tmp/ipython-input-2544992276.py:289: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot = gr.Chatbot(
/tmp/ipython-input-2544992276.py:289: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


🚀 Launching Enhanced Plant Disease AI Assistant...
⚡ Powered by Groq for ultra-fast AI responses!
📱 Share the public URL below!

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2c753eb3e94bfc00e6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: [['input_layer']]
Received: inputs=Tensor(shape=(1, 224, 224, 3))
  warnings.warn(msg)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2187, in process_api
    inputs = await self.preprocess_data(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1829, in preprocess_data
    inputs_cached = await processing_utils.a

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://2c753eb3e94bfc00e6.gradio.live


## 🎉 Demo is Running!

**Instructions:**
1. Click the public Gradio URL above
2. Upload a leaf image
3. Click "Analyze Disease"
4. Get disease detection + treatment recommendations
5. Click "Get Research Papers" for latest research

**Note:** The public URL stays active for 72 hours!